# Binning (Discretization)

Binning converts a **continuous numeric feature** into **discrete categories (bins)**. It splits into three families based on *how* the bin boundaries are decided:

```
                        Binning
                           |
        ┌──────────────────┼──────────────────┐
        |                  |                   |
  Unsupervised          Supervised           Custom
        |                  |
   ├─ Equal Width      └─ Decision Tree
   ├─ Equal Frequency      Binning
   └─ K-Means
```

---

## 1. Unsupervised Binning

**What it is:** Bin boundaries are decided **without looking at the target variable** — purely from the shape/spread of the feature itself. No learning of a target relationship happens.

**Common variants:**
| Method | How it works | Use case |
|---|---|---|
| Equal Width | Splits the range into N bins of **equal width** — `width = (max-min)/N` | Roughly uniform data |
| Equal Frequency | Splits so each bin holds **~equal number of observations** (quantile-based) | Skewed data |
| K-Means | Clusters values into N groups; boundaries = **midpoints between cluster centers** | Data with natural groupings/density |

```
EQUAL WIDTH (5 bins, same width each) — on skewed Income data
---------------------------------------------------------------
[   5.0 -   54.0) | ███████████████████████████████████ 1573
[  54.0 -  103.0) | ████████ 365
[ 103.0 -  152.0) | █ 49
[ 152.0 -  201.0) |  9
[ 201.0 -  250.0) |  4

EQUAL FREQUENCY (5 bins, same count each)
---------------------------------------------------------------
[   5.0 -   20.8) | ███████████████████████████████████ 400
[  20.8 -   29.3) | ███████████████████████████████████ 400
[  29.3 -   39.3) | ███████████████████████████████████ 400
[  39.3 -   55.6) | ███████████████████████████████████ 400
[  55.6 -  250.0) | ███████████████████████████████████ 400

K-MEANS (5 clusters, boundaries = midpoints between centers)
---------------------------------------------------------------
[   5.0 -   29.6) | ███████████████████████████████████ 811
[  29.6 -   49.9) | █████████████████████████████ 689
[  49.9 -   78.1) | ██████████████ 327
[  78.1 -  126.3) | ██████ 146
[ 126.3 -  250.0) | █ 27
Cluster centers: 20.4, 38.8, 61.1, 95.0, 157.6
```

**Note:** Equal Width is **very sensitive to outliers** (max/min decide the range, so skewed data piles into one bin). Equal Frequency and K-Means both adapt to the skew instead of ignoring it — Equal Frequency forces equal *counts*, K-Means follows equal *density*.

---

## 2. Supervised Binning (Decision Tree Binning)

**What it is:** A shallow Decision Tree (`max_depth = 2–4`) is trained using the feature (X) against the **target (y)**. The tree's split thresholds become the bin boundaries — so the bins are chosen to maximize information about the target, not just to describe the feature's own shape.

```
DECISION TREE BINNING (splits chosen using target variable)
-------------------------------------------------------------
[   5.0 -   27.5) | ███████████████████████████████████ 718
[  27.5 -   44.0) | ██████████████████████████████ 626
[  44.0 -   55.7) | ████████████ 257
[  55.7 -  250.0) | ███████████████████ 399

Target default-rate per bin -> bin0: 0.81 | bin1: 0.61 | bin2: 0.33 | bin3: 0.09
```
The boundaries (27.5, 44.0, 55.7) weren't derived from income's own distribution at all — they were picked because they best separate high-risk from low-risk incomes. That's why the default rate drops cleanly and monotonically bin by bin.

**Key properties:**
- Bins maximize **information gain / impurity reduction** w.r.t. the target
- Needs a **target variable** — only usable in supervised settings
- Risk of **overfitting** if the tree is too deep → keep `max_depth` small
- More computationally expensive than the unsupervised methods

---

## 3. Custom Binning

**What it is:** Bin edges are set **manually** using domain or business knowledge — no algorithm or statistic decides them.

```
CUSTOM BINNING (manual, domain-driven — e.g. income tax slabs)
----------------------------------------------------------------
[    0 -   25 ) Low income
[   25 -   60 ) Middle income
[   60 -  120 ) Upper-middle
[  120 -  250+) High income
```

**Key properties:**
- Fully **interpretable**, aligned with real-world meaning (policy, business rules)
- **No statistical basis** — doesn't adapt automatically to new data
- Only as good as the domain knowledge behind it

---

## Quick comparison

| Method | Uses Target? | Outlier Handling | Bin Population | Best for |
|---|:---:|:---:|---|---|
| Equal Width | No | Poor | Unequal | Uniformly distributed data |
| Equal Frequency | No | Good | Equal | Skewed data |
| K-Means | No | Moderate | Density-based | Naturally clustered data |
| Decision Tree | Yes | Good | Target-driven | Predictive modeling / classification |
| Custom | No | N/A | Domain-defined | Business/domain-specific rules |

Before Binning

In [ ]:

import numpy as np
import pandas as pd
from pandas import set_option

import  matplotlib.pyplot as plt
import seaborn as sns

set_option('display.max_columns', None)


In [ ]:
try:
    df = pd.read_csv(r'/home/mukul/Main/Basic_DataScience-4ML/assets/data/titanic.csv',
                     usecols=[ 'Survived','Age','Fare'],
                     header=0,
                     engine='python',
                     on_bad_lines='error'

                     )
except Exception as e:
    print(e)

In [ ]:
df.value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(10,10))

plt.subplot(2,2,1)
sns.boxplot(data=df, x='Fare')

plt.subplot(2,2,2)
sns.boxplot(data=df, x='Age')

plt.subplot(2,2,3)
sns.boxplot(data=df, x='Survived')


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.compose import  ColumnTransformer

from sklearn.preprocessing import KBinsDiscretizer, Binarizer,
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier

In [ ]:
x = df.drop('Survived', axis=1)
y = df['Survived']
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=7)

In [ ]:
num_1 = make_pipeline(
    SimpleImputer(strategy='mean'),
)

In [ ]:
cr = ColumnTransformer(
    transformers=[
        ('num', num_1, ['Fare'])
    ]
)

In [ ]:
model = make_pipeline(
    cr,
    DecisionTreeClassifier()
)

In [ ]:
model.fit(x_train,y_train)

In [ ]:
model.predict(x_test)
y_pred = model.predict(x_test)
print(accuracy_score(y_test,y_pred))

using binning


In [ ]:
x = df.drop('Survived', axis=1)
y = df['Survived']
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=7)


In [ ]:
num_1 = make_pipeline(
    SimpleImputer(strategy='mean'),
)
num_2 = make_pipeline(
    SimpleImputer(strategy='mean'),
    KBinsDiscretizer(
        strategy='quantile',
        n_bins=20,
        encode='ordinal'
    )
)

In [ ]:
cr = ColumnTransformer(
    transformers=[
        ('num', num_1, ['Age']),
        ('num_1', num_2, ['Age', 'Fare'])
    ]
)

In [ ]:
model = make_pipeline(
    cr,
    DecisionTreeClassifier()
)

In [ ]:
try:
    model.fit(x_train,y_train)
except Exception as e:
    print(e)

In [ ]:
model.predict(x_test)
y_pred = model.predict(x_test)
print(accuracy_score(y_test,y_pred))

4. Binarization

What it is: The extreme case of binning with just 2 bins — every value is converted to 0 or 1 based on a single threshold. Anything ≤ threshold → 0, anything > threshold → 1 (sklearn.preprocessing.Binarizer).

value > threshold  ->  1
value <= threshold ->  0
BINARIZATION (threshold = 40.0, on Income data)
----------------------------------------------------
[   5.0 -  40.0]  -> 0  | ███████████████████████████████████ 1224
(  40.0 - 250.0]  -> 1  | ██████████████████████ 776

Where it's used:

Domain	Example
* Text / NLP	Bag-of-Words → convert word counts to word presence/absence (occurred or not, ignoring frequency)

* Image processing	Pixel intensity → black/white thresholding (e.g. document scanning, edge masks)

* Recommender systems	"Purchased / rated" (1) vs "never interacted" (0), ignoring how many times

* Feature engineering	Turning a noisy continuous signal into a simple flag feature (e.g. is_high_spender, has_outlier_reading)

* Anomaly / rule-based systems	Sensor value above/below a safety threshold → alarm (1) or normal (0)

Key properties:

Simplest possible discretization — only 1 threshold, only 2 bins
Maximum information loss of any binning method (throws away all magnitude info, keeps only above/below)
Threshold is usually domain-chosen (like custom binning) rather than learned from data
Very fast, very interpretable — often used as a quick binary flag, not a full model feature by itself

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Binarizer

pipe = make_pipeline(Binarizer(threshold=40.0))